<a href="https://colab.research.google.com/github/ShiYu0318/ShiYu-AI-Courses-Archive/blob/main/MNIST_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 下載套件

In [ ]:
!pip install gradio

In [ ]:
%matplotlib inline

import numpy as np    # 矩陣與數值運算
import matplotlib.pyplot as plt    # 繪圖
from ipywidgets import widgets, interact_manual, IntSlider, Button, HBox, VBox    # 建立互動 UI
from IPython.display import display    # 顯示 widget
from PIL import Image    # 影像處理
import gradio as gr    # 建立 Web UI

import tensorflow as tf    # 深度學習框架
from tensorflow.keras.datasets import mnist    # MNIST 資料集
from tensorflow.keras.utils import to_categorical    # one-hot encoding
from tensorflow.keras.models import Sequential    # 線性模型
from tensorflow.keras.layers import Dense, Dropout    # 神經網路層
from tensorflow.keras.optimizers import SGD    # 優化器
from sklearn.metrics import confusion_matrix    # 混淆矩陣
import seaborn as sns    # 視覺化 heatmap

# 2. 載入資料集與預處理

In [ ]:
# TODO 1

# 3. 建構神經網路

In [ ]:
# TODO 2

# 4. 訓練模型

In [ ]:
# TODO 3

# 5. 畫出 Loss 和 Accuracy 變化圖

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))    # 建立左右兩圖

axes[0].plot(history.history['loss'], marker='o')    # 訓練 loss
axes[0].plot(history.history['val_loss'], marker='o')    # 驗證 loss
axes[0].set_title('Model Loss')    # 標題
axes[0].set_xlabel('Epoch')    # x 軸
axes[0].set_ylabel('Loss')    # y 軸
axes[0].legend(['Train', 'Validation'])    # 圖例
axes[0].grid(True)    # 格線

axes[1].plot(history.history['accuracy'], marker='o')    # 訓練 accuracy
axes[1].plot(history.history['val_accuracy'], marker='o')    # 驗證 accuracy
axes[1].set_title('Model Accuracy')    # 標題
axes[1].set_xlabel('Epoch')    # x 軸
axes[1].set_ylabel('Accuracy')    # y 軸
axes[1].legend(['Train', 'Validation'])    # 圖例
axes[1].grid(True)    # 格線

plt.tight_layout()    # 自動排版
plt.show()    # 顯示圖

# 6. 預測與評估

In [ ]:
# TODO 4

In [ ]:
print('loss:', loss)    # 顯示 loss
print(f"測試資料正確率 {acc*100:.2f}%")    # 顯示 accuracy

plt.figure(figsize=(6,5))    # 設定大小
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')    # 畫 heatmap
plt.xlabel('Predicted')    # x 軸
plt.ylabel('True')    # y 軸
plt.title('Confusion Matrix')    # 標題
plt.show()    # 顯示圖

# 7. 辨識結果可視化

In [ ]:
ROWS = 2    # 幾列
COLS = 8    # 每列幾張

true_labels = np.argmax(y_test, axis=1)    # one-hot 轉數字

correct_indices = np.where(predict == true_labels)[0]    # 找預測正確
wrong_indices = np.where(predict != true_labels)[0]    # 找預測錯誤


def plot_panel(indices, title):    # 畫圖片面板
    fig, axes = plt.subplots(ROWS, COLS, figsize=(1.5*COLS, 1.5*ROWS))    # 建畫布
    axes = np.array(axes).reshape(ROWS, COLS)    # 保證 2D

    fig.suptitle(title, fontsize=16)    # 標題

    for i in range(ROWS * COLS):    # 逐格畫
        r, c = divmod(i, COLS)    # 算位置
        ax = axes[r, c]    # 取子圖

        if i < len(indices):    # 有資料才畫
            idx = indices[i]    # 取 index
            ax.imshow(x_test[idx].reshape(28,28), cmap='Greys')    # 顯示圖片
            ax.axis('off')    # 關閉座標

            pred = predict[idx]    # 預測值
            true = true_labels[idx]    # 真實值

            ax.set_title(    # 顯示資訊
                f"ID:{idx}\nPred:{pred} True:{true}",
                fontsize=10
            )
        else:
            ax.axis('off')    # 空格關閉

    plt.tight_layout()    # 自動排版
    plt.show()    # 顯示


def show_dashboard():    # 主函式
    total = ROWS * COLS    # 總張數

    correct_sample = np.random.choice(    # 抽正確樣本
        correct_indices,
        min(total, len(correct_indices)),
        replace=False
    )

    wrong_sample = np.random.choice(    # 抽錯誤樣本
        wrong_indices,
        min(total, len(wrong_indices)),
        replace=False
    )

    plot_panel(correct_sample, "Correct Predictions")    # 上面：正確
    plot_panel(wrong_sample, "Wrong Predictions")    # 下面：錯誤

interact_manual(show_dashboard)    # 建立按鈕

# 8. 透過 Gradio 快速建立互動式 WebUI，用自己訓練的模型辨識自己手寫的數字

In [ ]:
def resize_image(inp):
    image = np.array(inp["layers"][0], dtype=np.float32)    # 取畫板資料
    image = image.astype(np.uint8)    # 轉型

    image_pil = Image.fromarray(image)    # 轉 PIL

    background = Image.new("RGB", image_pil.size, (255, 255, 255))    # 建白底
    background.paste(image_pil, mask=image_pil.split()[3])    # 貼上透明圖
    image_pil = background    # 更新

    image_gray = image_pil.convert("L")    # 轉灰階
    img_array = np.array(image_gray.resize((28, 28), resample=Image.LANCZOS))    # resize

    img_array = 255 - img_array    # 反轉顏色（符合 MNIST）
    img_array = img_array.reshape(1, 784) / 255.0    # 攤平 + 正規化

    return img_array    # 回傳


def recognize_digit(inp):
    img_array = resize_image(inp)    # 預處理

    prediction = model.predict(img_array).flatten()    # 預測機率
    labels = list('0123456789')    # 標籤

    return {labels[i]: float(prediction[i]) for i in range(10)}    # 回傳結果


iface = gr.Interface(
    fn=recognize_digit,
    inputs=gr.Sketchpad(),
    outputs=gr.Label(num_top_classes=3),
    title="MNIST 手寫辨識",
    description="請在畫板上繪製數字"
)    # 建立介面

iface.launch(share=True, debug=True)    # 啟動服務